In [ ]:
#0 Load Libraries and Configurations

import os
import wrds
import pandas as pd
import numpy as np

# ---------- User-configurable paths ----------
PATH_DATA_INTERMEDIATE = "/Users/nglei/Desktop/Academics/SMU/Modules/QF600 Asset Pricing/Project/Project Code/cz_data/intermediate"  # <-- change this
os.makedirs(PATH_DATA_INTERMEDIATE, exist_ok=True)

OUT_PARQUET = os.path.join(PATH_DATA_INTERMEDIATE, "m_CRSPAcquisitions.parquet")
OUT_CSV     = os.path.join(PATH_DATA_INTERMEDIATE, "m_CRSPAcquisitions.csv")

In [ ]:
#1 Load CRSP Data

SQL = """
SELECT
    a.permno,
    a.distcd,
    a.exdt,
    a.acperm
FROM crsp.msedist AS a
WHERE a.exdt >= DATE '2000-01-01';
"""

In [ ]:
#2 CRSP Data Extraction From WRDS

db = wrds.Connection()
df = db.raw_sql(SQL, date_cols=["exdt"])

In [ ]:
#3 Data Cleaning

# ---------------- Keep valid acperm & non-missing ex-date ----------------
# Stata: keep if acperm > 999 & acperm < . ; drop if missing(time_d)
df = df[(df["acperm"].notna()) & (df["acperm"] > 999)].copy()
df = df.dropna(subset=["exdt"])

# ---------------- Create monthly timestamp from ex-date ----------------
# Stata: gen time_avail_m = mofd(time_d); format %tm; (then drops time_d)
df["time_avail_m"] = df["exdt"].dt.to_period("M").dt.to_timestamp("MS")

# ---------------- Turn into list of PERMNOs created in spinoffs ----------------
# Stata:
#   gen SpinoffCo = 1
#   drop permno
#   rename acperm permno
#   keep permno SpinoffCo
#   duplicates drop
out = df[["acperm"]].copy()
out = out.rename(columns={"acperm": "permno"})
out["SpinoffCo"] = 1
out = out.drop_duplicates(subset=["permno"]).reset_index(drop=True)

# (Optional) sort for readability
out = out.sort_values("permno", kind="mergesort").reset_index(drop=True)

# ---------------- Save ----------------
out.to_parquet(OUT_PARQUET, index=False)
out.to_csv(OUT_CSV, index=False)

print("Saved:")
print(" -", OUT_PARQUET)
print(" -", OUT_CSV)
print(out.head())